# Genetic algorithms — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sphere(x, target=None):
    x=np.asarray(x, dtype=float)
    if target is None: target=np.zeros_like(x)
    return float(np.sum((x-np.asarray(target,dtype=float))**2))
def rastrigin(x):
    x=np.asarray(x,dtype=float); return float(10*len(x)+np.sum(x*x-10*np.cos(2*np.pi*x)))
def softmax_min(vals, temp=1.0):
    vals=np.asarray(vals,dtype=float); w=np.exp(-(vals-vals.min())/temp); return w/w.sum()
print('setup ok')

## Selection, recombination, mutation, and elitism on a small numeric fitness landscape
A genetic algorithm keeps a population of candidate solutions, gives better candidates more chances to reproduce, recombines them, mutates them, and preserves progress through elitism. These five cells make each moving part visible on tiny NumPy landscapes.

In [ ]:
# 1) Fitness-proportionate selection on a 1D landscape
pop=np.array([0.,2.,4.,6.])
fitness=-(pop-3)**2+10
weights=fitness-fitness.min()+1
probs=weights/weights.sum()
plt.figure(figsize=(6,3)); plt.bar([str(x) for x in pop], probs, color='tab:blue')
plt.xlabel('candidate x'); plt.ylabel('selection probability'); plt.title('better fitness gets more reproduction tickets')
assert np.allclose(probs, [0.05,0.45,0.45,0.05])

In [ ]:
# 2) Crossover averages two good parents, mutation explores nearby
parents=np.array([2.,4.])
child=parents.mean()+0.5
fchild=-(child-3)**2+10
xs=np.linspace(0,6,200); ys=-(xs-3)**2+10
plt.figure(figsize=(6,3)); plt.plot(xs,ys); plt.scatter(parents, -(parents-3)**2+10, label='parents'); plt.scatter([child],[fchild],c='r',label='child')
plt.legend(); plt.xlabel('x'); plt.ylabel('fitness'); plt.title('recombine 2 and 4, mutate to 3.5')
assert abs(child-3.5)<1e-12 and abs(fchild-9.75)<1e-12

In [ ]:
# 3) A tiny GA improves best and mean fitness over generations
rng=np.random.default_rng(1); pop=rng.uniform(-2,7,20); best=[]; mean=[]
for _ in range(18):
    fit=-(pop-3)**2+10; best.append(fit.max()); mean.append(fit.mean())
    elite=pop[np.argmax(fit)]; probs=(fit-fit.min()+1); probs=probs/probs.sum()
    parents=rng.choice(pop, size=(19,2), p=probs); children=parents.mean(axis=1)+rng.normal(0,0.25,19)
    pop=np.r_[elite, children]
plt.figure(figsize=(6,3)); plt.plot(best,label='best'); plt.plot(mean,label='mean'); plt.axhline(10,ls='--',c='k'); plt.legend(); plt.xlabel('generation'); plt.ylabel('fitness')
plt.title('selection plus variation climbs the landscape')
assert best[-1] > best[0] and mean[-1] > mean[0]

In [ ]:
# 4) Mutation scale trades exploitation for exploration
rng=np.random.default_rng(2); sigmas=[0.02,0.2,1.0]; finals=[]; diversity=[]
for sigma in sigmas:
    pop=rng.uniform(-2,7,30)
    for _ in range(20):
        fit=-(pop-3)**2+10; probs=(fit-fit.min()+1); probs=probs/probs.sum()
        parents=rng.choice(pop, size=(30,2), p=probs); pop=parents.mean(axis=1)+rng.normal(0,sigma,30)
    finals.append((-(pop-3)**2+10).mean()); diversity.append(pop.std())
plt.figure(figsize=(6,3)); plt.plot(sigmas, finals, '-o', label='mean fitness'); plt.twinx(); plt.plot(sigmas, diversity, '-s', color='tab:orange', label='diversity')
plt.xscale('log'); plt.title('mutation knob: search radius versus convergence')
assert diversity[-1] > diversity[0] and finals[1] >= finals[0]-0.5

In [ ]:
# 5) In 2D, the population cloud converges toward the optimum
rng=np.random.default_rng(3); target=np.array([1.,-1.]); pop=rng.uniform(-4,4,(40,2)); start=pop.copy()
for _ in range(25):
    fit=-np.sum((pop-target)**2, axis=1); elite=pop[np.argmax(fit)]; probs=(fit-fit.min()+1); probs=probs/probs.sum()
    p=rng.choice(len(pop), size=(39,2), p=probs); pop=(pop[p[:,0]]+pop[p[:,1]])/2+rng.normal(0,0.18,(39,2)); pop=np.vstack([elite,pop])
plt.figure(figsize=(4,4)); plt.scatter(start[:,0],start[:,1],alpha=.25,label='start'); plt.scatter(pop[:,0],pop[:,1],alpha=.8,label='final'); plt.scatter([1],[-1],c='r',marker='*',s=120,label='optimum')
plt.legend(); plt.axis('equal'); plt.title('population convergence in 2D')
assert np.linalg.norm(pop.mean(axis=0)-target) < np.linalg.norm(start.mean(axis=0)-target)